# Whisper paraphasia detection — Scripts-Fridriksson LOSO on Kaggle (2× T4)

Data comes from the private Kaggle dataset `evandiewald/aphasia-scripts-fridriksson` (fetched with kagglehub, no need to attach it).
Needs the `WANDB_API_KEY` Kaggle secret and internet on.

Two folds train at once, one per GPU, in a background process that keeps running if this notebook disconnects.
Each finished fold is uploaded as a W&B artifact (`<RUN_NAME>-fold_<spk>`); re-running the launch cell in a new session skips uploaded folds.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess

REPO_DIR = '/kaggle/working/aphasia-modeling'
RUN_NAME = 'scripts-small-taginit'
CKPT = f'/root/checkpoints/{RUN_NAME}'  # / has ~1 TB; /kaggle/working is capped at 20 GB
RESULTS = f'/kaggle/working/results/{RUN_NAME}'
DATA_PATH = 'datasets/Scripts/scripts_fridriksson.json'
os.makedirs(CKPT, exist_ok=True)

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/evandiewald/aphasia-modeling.git {REPO_DIR}
!cd {REPO_DIR} && git log --oneline -1

In [ ]:
!cd {REPO_DIR} && pip install -e . -q

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')

In [ ]:
# Link the dataset into the repo at the paths the JSON references
import json, kagglehub
src = kagglehub.dataset_download('evandiewald/aphasia-scripts-fridriksson')
audio_dir = f'{REPO_DIR}/datasets/Scripts/Fridriksson/audio'
os.makedirs(os.path.dirname(audio_dir), exist_ok=True)
if not os.path.islink(audio_dir):
    os.symlink(f'{src}/audio', audio_dir)
!cp {src}/scripts_fridriksson.json {REPO_DIR}/{DATA_PATH}

records = json.load(open(f'{REPO_DIR}/{DATA_PATH}'))
missing = {r['audio_path'] for r in records if not os.path.exists(f"{REPO_DIR}/{r['audio_path']}")}
print(f"{len(records)} utterances, {len({r['speaker_id'] for r in records})} speakers, {len(missing)} missing audio files")
assert not missing, sorted(missing)[:5]

In [ ]:
# Launch all folds in the background, one per GPU (T4: fp16, batch 4 x accum 4 = effective 16)
TAG_CLASSES = 'pns'  # 'pn' to leave [s] to Stage 2
if subprocess.run(['pgrep', '-f', 'run_loso_parallel'], capture_output=True).returncode == 0:
    print('Runner already running')
else:
    cmd = (f'cd {REPO_DIR} && nohup python scripts/run_loso_parallel.py --gpus 0,1 '
           f'--data_path {DATA_PATH} --output_dir {CKPT} --run_name {RUN_NAME} --wandb_artifacts -- '
           f'--tag_classes {TAG_CLASSES} --fp16 --batch_size 4 --grad_accum 4 --num_workers 2 --wandb '
           f'>> {CKPT}/runner.log 2>&1 &')
    subprocess.run(cmd, shell=True, start_new_session=True)
    print(f'Started; runner log: {CKPT}/runner.log')

In [ ]:
# Progress: re-run any time
import re, glob
print(open(f'{CKPT}/runner.log').read()[-2000:])
for log in sorted(glob.glob(f'{CKPT}/logs/*.log')):
    evals = re.findall(r"\{'eval_loss'.*?'epoch': '[\d.]+'\}", open(log).read())
    print(os.path.basename(log), evals[-1] if evals else '(no eval yet)')
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv

In [ ]:
# After 'All folds complete': fetch every fold and run pooled LOSO evaluation
!cd {REPO_DIR} && python scripts/run_loso_parallel.py --download --data_path {DATA_PATH} --output_dir {CKPT} --run_name {RUN_NAME}
!cd {REPO_DIR} && python scripts/evaluate_model.py --model_dir {CKPT} --data_path {DATA_PATH} \
    --loso --output_dir {RESULTS} --batch_size 16